Setup stuff

In [ ]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
import scipy.stats as st
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("domestic_well_radon.csv")
df = df.dropna(subset=["log_radon", "well_depth_ft", "lithology", "aquifer"])
df["log_depth"] = np.log(df["well_depth_ft"])
df["centered_log_depth"] = (df["log_depth"] - df["log_depth"].mean())
df = df[["log_radon", "centered_log_depth", "lithology", "aquifer"]]
df.dropna()
print("Number of lithologies:", df["lithology"].nunique())
print("Number of aquifers:", df["aquifer"].nunique())
print("Wells with complete data:", len(df))
print(df.groupby("lithology")["log_radon"].agg(["count", "mean", "std", "min", "max"]))
df.head()

Priors (change later):

In [ ]:
intercept = {
    "Intercept": bmb.Prior("Normal", mu=6.0, sigma=2.0)
}

depth = {
    "centered_log_depth": bmb.Prior("Normal", mu=0.0, sigma=1.0),
}

sigma = {
    "sigma": bmb.Prior("HalfNormal", sigma=1.0)
}

lithology_intercept = {
    "1|lithology": bmb.Prior("Normal", mu=0.0, sigma=bmb.Prior("HalfNormal", sigma=1.0))
}

aquifier_intercept = {
    "1|aquifer": bmb.Prior("Normal", mu=0.0, sigma=bmb.Prior("HalfNormal", sigma=1.0)),
}

varying_sigma_prior = {
    "sigma": {
        "lithology": bmb.Prior("Normal", mu=0.0, sigma=1.0),
    }
}
lithology_specific_depth =  {
    "centered_log_depth|lithology": bmb.Prior(
        "Normal", mu=0.0, sigma=bmb.Prior("HalfNormal", sigma=0.5)
    )
}

nu = {
    "nu": bmb.Prior("Gamma", alpha=2, beta=0.1)
}

pooled_priors = intercept | depth | sigma

aquifer_hierarchical_priors = intercept | depth | sigma | aquifier_intercept

combined_hierarchical_priors = intercept | depth | sigma | lithology_intercept | aquifier_intercept

combined_varying_sigma_hierarchical_priors = intercept | depth | lithology_intercept | aquifier_intercept | varying_sigma_prior
 
lithology_specific_depth_coefficient_priors = intercept | depth | sigma | lithology_specific_depth | lithology_intercept | aquifier_intercept

combined_varying_sigma_hierarchical_priors_t = intercept | depth | lithology_intercept | aquifier_intercept | varying_sigma_prior | nu

pooled_priors_t = intercept | depth | sigma | nu

Model. Hierarchical, with intercept from lithology & aquifer

In [ ]:
combined_hierarchical_model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (1 | lithology) + (1 | aquifer)",
    ),
    data=df,
    priors=combined_hierarchical_priors,
)

combined_varying_sigma_hierarchical_model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (1 | lithology) + (1 | aquifer)",
        "sigma ~ 0 + lithology",
    ),
    data=df,
    priors=combined_varying_sigma_hierarchical_priors,
)


combined_varying_sigma_hierarchical_model_t = bmb.Model(
    bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (1 | lithology) + (1 | aquifer)",
        "sigma ~ 0 + lithology",
    ),
    data=df,
    family="t",
    priors=combined_varying_sigma_hierarchical_priors_t,
)
aquifer_hierarchical_model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (1 | aquifer)",
    ),
    data=df,
    priors=aquifer_hierarchical_priors,
)

lithology_specific_depth_coefficient_model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (centered_log_depth | lithology) + (1 | lithology) + (1 | aquifer)",
    ),
    data=df,
    priors=lithology_specific_depth_coefficient_priors,
)

pooled_model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth",
    ),
    data=df,
    priors=pooled_priors,
)

pooled_model_t = bmb.Model(
    bmb.Formula("log_radon ~ 1 + centered_log_depth"),
    data=df,
    family="t",
    priors=pooled_priors_t,
)

In [ ]:
combined_hierarchical_fit = combined_hierarchical_model.fit(target_accept=0.95, idata_kwargs={"log_likelihood": True})


In [ ]:
combined_varying_sigma_hierarchical_fit = combined_varying_sigma_hierarchical_model.fit(target_accept=0.95, idata_kwargs={"log_likelihood": True})


In [ ]:
combined_varying_sigma_hierarchical_fit_t = combined_varying_sigma_hierarchical_model_t.fit(target_accept=0.95, idata_kwargs={"log_likelihood": True})

In [ ]:
aquifer_hierarchical_fit =  aquifer_hierarchical_model.fit(draws=4000, target_accept=0.8, idata_kwargs={"log_likelihood": True})

In [ ]:
lithology_specific_depth_coefficient_fit =  lithology_specific_depth_coefficient_model.fit(target_accept=0.95, idata_kwargs={"log_likelihood": True})

In [ ]:
pooled_fit = pooled_model.fit(target_accept=0.8, idata_kwargs={"log_likelihood": True})

In [ ]:
pooled_fit_t = pooled_model_t.fit(target_accept=0.8, idata_kwargs={"log_likelihood": True})

Summary statistics:

In [ ]:
fits_and_models = {
    "pooled": (pooled_model, pooled_fit),
    "pooled_t": (pooled_model_t, pooled_fit_t),
    "aquifer_hierarchical": (aquifer_hierarchical_model, aquifer_hierarchical_fit),
    "combined_hierarchical": (combined_hierarchical_model, combined_hierarchical_fit),
    "lithology_specific_depth_coefficient": (lithology_specific_depth_coefficient_model,
                                             lithology_specific_depth_coefficient_fit),
    "combined_varying_sigma_hierarchical": (combined_varying_sigma_hierarchical_model,
                                            combined_varying_sigma_hierarchical_fit),
    "combined_varying_sigma_hierarchical_t": (combined_varying_sigma_hierarchical_model_t,
                                              combined_varying_sigma_hierarchical_fit_t),
}

fits_and_models_good = {
    "combined_varying_sigma_hierarchical": (combined_varying_sigma_hierarchical_model,
                                            combined_varying_sigma_hierarchical_fit),
    "combined_varying_sigma_hierarchical_t": (combined_varying_sigma_hierarchical_model_t,
                                              combined_varying_sigma_hierarchical_fit_t),
}

common_vars = ["Intercept", "centered_log_depth", "sigma"]

models = {
    "combined_hierarchical": (
        combined_hierarchical_fit,
        common_vars + ["1|lithology_sigma", "1|aquifer_sigma"],
    ),
    "combined_varying_sigma_hierarchical": (
        combined_varying_sigma_hierarchical_fit,
        ["Intercept", "centered_log_depth", "sigma_lithology", "1|lithology_sigma", "1|aquifer_sigma"],
    ),
    "combined_varying_sigma_hierarchical_t": (
        combined_varying_sigma_hierarchical_fit_t,
        ["Intercept", "centered_log_depth", "sigma_lithology", "1|lithology_sigma", "1|aquifer_sigma", "nu"],
    ),
    "aquifer_hierarchical": (
        aquifer_hierarchical_fit,
        common_vars + ["1|aquifer_sigma"],
    ),
    "lithology_specific_depth_coefficient": (
        lithology_specific_depth_coefficient_fit,
        common_vars + ["centered_log_depth|lithology_sigma", "1|lithology_sigma", "1|aquifer_sigma"],
    ),
    "pooled": (
        pooled_fit,
        common_vars,
    ),
    "pooled_t": (
        pooled_fit_t,
        common_vars + ["nu"],
    ),
}

In [ ]:
import arviz as az

def analyze_fit(name, fit, var_names):
    print(f"=== {name} ===")
    print("Divergences:", int(fit.sample_stats["diverging"].sum()))
    print(az.summary(fit, var_names=var_names))

    loo = az.loo(fit, pointwise=True)
    print(f"LOO: {name}")
    print(loo)
    print()
    return loo


def compare_models(fits):
    print("=== Model comparison ===")
    comparison = az.compare(fits)

    comparison["diff_over_dse"] = comparison["elpd_diff"] / comparison["dse"]

    print(comparison[["rank", "elpd_loo", "p_loo", "elpd_diff", "dse",
                      "diff_over_dse", "weight", "warning"]])

    az.plot_compare(comparison, plot_ic_diff=True, legend=True)
    return comparison

common_vars = ["Intercept", "centered_log_depth", "sigma"]


#loos = {name: analyze_fit(name, fit, var_names) for name, (fit, var_names) in models.items()}

comparison = compare_models({name: fit for name, (fit, _) in models.items()})


Add some predictive checks.

In [ ]:
def plot_predictive_checks(fits, y_name="log_radon", obs_idx=None,
                           ci_prob=0.95, num_pp_samples=100, title=None):
    """KDE, PIT-ECDF and PIT-ECDF difference for several fits, one row per model.

    fits: dict {name: InferenceData with posterior_predictive and log_likelihood}
    obs_idx: optional array of observation positions, e.g. one lithology
    title: optional figure title, e.g. the lithology name
    """
    fig, axes = plt.subplots(len(fits), 3, figsize=(15, 3.6 * len(fits)), squeeze=False)
    eval_points = np.linspace(0, 1, 101)
    coords = {"__obs__": obs_idx} if obs_idx is not None else None

    for row, (name, fit) in enumerate(fits.items()):
        az.plot_ppc(fit, var_names=[y_name], coords=coords,
                    num_pp_samples=num_pp_samples, ax=axes[row, 0])

        pit = np.asarray(az.loo_pit(fit, y=y_name)).ravel()
        if obs_idx is not None:
            pit = pit[obs_idx]

        for ax, difference in zip(axes[row, 1:], [False, True]):
            az.plot_ecdf(pit, cdf=st.uniform().cdf, rvs=st.uniform().rvs, eval_points=eval_points,
                         confidence_bands="simulated", ci_prob=ci_prob,
                         difference=difference, ax=ax)

        axes[row, 0].set_title(f"{name}: KDE")
        axes[row, 1].set_title("PIT-ECDF")
        axes[row, 2].set_title("PIT-ECDF difference")

    if title is not None:
        fig.suptitle(title, fontsize=16)
        fig.tight_layout(rect=(0, 0, 1, 0.97))   # leave room for the title
    else:
        fig.tight_layout()
    plt.show()
    return fig


fits_good = {name: fit for name, (model, fit) in fits_and_models_good.items()}

for lith in df["lithology"].unique():
    idx = np.where(df["lithology"].values == lith)[0]
    plot_predictive_checks(fits_good, obs_idx=idx, title=f"{lith} (n = {len(idx)})");


In [ ]:
def convergence_summary(fits, decimals=3):
    """Divergences, worst R-hat and smallest ESS over all parameters, per model."""
    rows = []
    for name, fit in fits.items():
        # skip per-observation quantities added by predict (mu, sigma per well)
        params = [v for v in fit.posterior.data_vars
                  if "__obs__" not in fit.posterior[v].dims]
        s = az.summary(fit, var_names=params, kind="diagnostics", round_to="none")
        rows.append({
            "model": name,
            "divergences": int(fit.sample_stats["diverging"].sum()),
            "max_rhat": s["r_hat"].max(),
            "max_rhat_param": s["r_hat"].idxmax(),
            "min_ess_bulk": s["ess_bulk"].min(),
            "min_ess_bulk_param": s["ess_bulk"].idxmin(),
            "min_ess_tail": s["ess_tail"].min(),
            "min_ess_tail_param": s["ess_tail"].idxmin(),
        })
    out = pd.DataFrame(rows).set_index("model")
    out["max_rhat"] = out["max_rhat"].round(decimals)
    out[["min_ess_bulk", "min_ess_tail"]] = out[["min_ess_bulk", "min_ess_tail"]].round(0).astype(int)
    return out

with pd.option_context("display.width", 200, "display.max_columns", None):
    print(convergence_summary({name: fit for name, (model, fit) in fits_and_models.items()}))